# 3. Computer Vision
following topics are provided by the syllabus:
1. [ ] Convolutional Layers
2. [ ] Image Classification
3. [ ] Object Detection (YOLO, SSD, DERT)
4. [ ] Image Segmentation (U-Net)
5. [ ] Pre-trained Vision Encoders (e.g. ResNet)
6. [ ] Image Augmentation
7. [ ] Generating Images with GANs
8. [ ] Self-Supervised Learning for Vision
9. [ ] Vision-text encoders (e.g. CLIP)
10. [ ] Diffusion Models

It becomes clear what Computer Vision entails by looking at the topics above. It is a field of study that focuses on enabling computers to interpret and understand visual information from the world, such as images and videos. The main component of computer vision is the use of convolutional layers. They have 2 very important properties: pattern recognition and local independence. Pattern recognition allows the model to identify and learn patterns in the input data, while local independence allows the model to recognize them regardless of their position in the input.

## Image classification
We start by creating a simple image classification model using a convolutional neural network (CNN). The model will be trained on a dataset of cats and dogs. The goal is to classify the images into different categories based on their content. We aim to cover topics 1, 2, 5, and 6 from the syllabus, as well as Autoencoders, Data Embeddigns, Pooling Techniques and Weight Initialization from the previous Chapter. The model will consist of several convolutional layers followed by fully connected layers. We will also apply image augmentation techniques to improve the model's performance and generalization ability.

### Data loading

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.functional as F
import torchvision
import torchvision.transforms as transforms

data_transforms = transforms.Compose([

    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

dataset = torchvision.datasets.ImageFolder(root='./PetImages', transform=data_transforms)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

### Convolutional Neural Network (CNN) Model

In [2]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5) # 3 input channels (RGB), 6 output channels, 5x5 kernel
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 16 channels, 5x5 feature map size
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2) # 2 output classes (cat and dog)
        self.relu = nn.modules.activation.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Training the Model

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print(device)

model = Net()
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

model.train()
for _ in range(10):
    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {_ + 1}, Loss: {running_loss / len(dataloader):.4f}')



cuda
